# Tutorial: Inspect and Start Run

Audience:
- You are iterating on Hypergraph workflows in a notebook and want a fast way to see what failed without wrapping every run in `try/except`.

Prerequisites:
- Run this notebook from the Hypergraph repo root.
- Install the repo environment with `uv sync --group dev`.

Learning goals:
- Use `inspect=True` to capture reusable inspect data.
- Compare blocking `run()` with background `start_run()`.
- Stop a background run before the next superstep starts.


In [ ]:
from __future__ import annotations

import asyncio
import time

from hypergraph import AsyncRunner, Graph, SyncRunner, node


@node(output_name="doubled")
def double(x: int) -> int:
    return x * 2


@node(output_name="tripled")
def triple(doubled: int) -> int:
    return doubled * 3


@node(output_name="boom")
def fail_after_double(doubled: int) -> int:
    raise ValueError("boom")


@node(output_name="doubled")
def slow_sync_seed(x: int) -> int:
    time.sleep(4.0)
    return x * 2


@node(output_name="tripled")
def slow_sync_triple(doubled: int) -> int:
    time.sleep(4.0)
    return doubled * 3


@node(output_name="final")
def finalize(tripled: int) -> int:
    return tripled + 1


@node(output_name="doubled")
async def slow_async_seed(x: int) -> int:
    await asyncio.sleep(4.0)
    return x * 2


@node(output_name="tripled")
async def slow_async_triple(doubled: int) -> int:
    await asyncio.sleep(4.0)
    return doubled * 3

## Blocking run with inspect

Use `error_handling="continue"` when you want a terminal `RunResult` object even if a node fails.


In [ ]:
graph = Graph([double, fail_after_double])

In [ ]:
graph

In [ ]:
runner = SyncRunner()

result = runner.run(graph, {"x": 5}, inspect=True, error_handling="continue")

In [ ]:
failure = result.failure
assert failure is not None

{
    "status": result.status.value,
    "failed_node": failure.node_name,
    "failed_inputs": failure.inputs,
    "double_outputs": result.view()["double"].outputs,
}

## Background start_run with live view

The handle gives you an object immediately. The next example uses two slow stages so you can actually watch the widget update before the run finishes.


In [ ]:
slow_graph = Graph([slow_sync_seed, slow_sync_triple, finalize])
run = runner.start_run(slow_graph, {"x": 5}, inspect=True)

In [ ]:
time.sleep(0.55)
live = run.view()

{
    "status": live.status,
    "nodes_visible": [node.node_name for node in live.nodes],
}

## Cooperative stop

The current node is allowed to finish; the next superstep will not start. In this staged example, that means `finalize` should never run.


In [ ]:
run.stop(info={"kind": "manual-test"})
stopped = run.result(raise_on_failure=False)

{
    "status": stopped.status.value,
    "values": stopped.values,
    "tripled_present": "tripled" in stopped.values,
    "final_present": "final" in stopped.values,
}

## Async example

Notebook cells support top-level `await`, so you can exercise the async handle directly with the same staged shape.


In [ ]:
async_runner = AsyncRunner()
async_graph = Graph([slow_async_seed, slow_async_triple, finalize])
async_run = async_runner.start_run(async_graph, {"x": 5}, inspect=True)

await asyncio.sleep(0.55)
before_stop = async_run.view().status
visible_nodes = [node.node_name for node in async_run.view().nodes]
async_run.stop(info={"kind": "manual-test"})
async_result = await async_run.result(raise_on_failure=False)

{
    "before_stop": before_stop,
    "visible_nodes": visible_nodes,
    "final_status": async_result.status.value,
    "tripled_present": "tripled" in async_result.values,
    "final_present": "final" in async_result.values,
}